In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# imports
import numpy as np
import pandas as pd
from pathlib import Path
from src.metrics.metrics import (
    organize_multiverse_results,
    within_bucket_variation,
    between_bucket_variation,
)
import matplotlib.pyplot as plt
from src.metrics.functional_anova import MultiverseFanova
from fanova import visualizer

# get all csv files in the data directory for run of interest
data_dir = Path("output/runs/1/data")
csv_files = list(data_dir.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files.")

# read all files with mixed types
arrays = []
for i, f in enumerate(csv_files):
    arr = np.genfromtxt(f, delimiter=',', dtype=None, encoding='utf-8')
    arrays.append(arr)
    if (i + 1) % 1000 == 0:
        print(f"Loaded {i+1}/{len(csv_files)} files")

# stack all arrays vertically
data = np.concatenate(arrays)
# convert data to pandas df
df = pd.DataFrame(data)
# remove all rows that are exact duplicates, since they are removing the rows of column names
df = df.drop_duplicates()
# make first row the header
df.columns = df.iloc[0]
df = df[1:].reset_index(drop=True)

In [ ]:
df.head()

In [ ]:
dimension_order = ["spell_gap_threshold",
 "new_job_notif_threshold",
 "new_job_no_notif_threshold",
 "missing_strategy",
 "split_seed",
 "model"]
# prepend mv_dim_ to all dimension names
dimension_order = ["mv_dim_" + dim for dim in dimension_order]
performance_metrics = ["acc", "f1", "roc_auc"]
# prepend test_ to all performance metric names
performance_metrics = ["test_" + metric for metric in performance_metrics]

In [ ]:
# make all cols that start with test_ numeric
for col in df.columns:
    if col.startswith("test_"):
        df[col] = pd.to_numeric(df[col])

In [ ]:
df = organize_multiverse_results(df, dimension_order, performance_metrics)

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
within = within_bucket_variation(df, "test_acc")

In [ ]:
within

In [ ]:
dimension_order

In [ ]:
# change keys of within to be the dimension names instead of the dimension columns
within = {dimension_order[i]: within[i] for i in range(len(dimension_order))}
within

In [ ]:
between = between_bucket_variation(df, "test_acc")

In [ ]:
within

In [ ]:
between

In [ ]:
# define all decision dimensions
decision_dims = {
    "mv_dim_missing_strategy": "Missing Strategy",
    "mv_dim_model": "Model",
    "mv_dim_new_job_no_notif_threshold": "New Job No-Notification Threshold",
    "mv_dim_new_job_notif_threshold": "New Job Notification Threshold",
    "mv_dim_spell_gap_threshold": "Spell Gap Threshold",
    "mv_dim_split_seed": "Train-Test Split Seed",
}

In [ ]:
# plot scatterplot of within vs between variation for each dimension
plt.figure(figsize=(10, 6))
for dim in dimension_order:
    plt.scatter(within[dim], between[dim], label=decision_dims[dim], s=100)
plt.xlabel("Within-Bucket Variation", fontsize=12)
plt.ylabel("Between-Bucket Variation", fontsize=12)
plt.title("Within vs Between Bucket Variation for Each Dimension", fontsize=14, fontweight='bold')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# get variance of test_acc across all leaves in the multiverse
overall_variance = df["test_acc"].var()
print(f"Overall Variance of Test Accuracy: {overall_variance}")

# get the sum of the within bucket variation across all dimensions
total_within_variation = sum(within.values())
print(f"Total Within-Bucket Variation: {total_within_variation}")

# get the sum of the between bucket variation across all dimensions
total_between_variation = sum(between.values())
print(f"Total Between-Bucket Variation: {total_between_variation}")

In [ ]:
# create a dataframe consisting of just the judgment call dimensions
# make them not multi-indexed, since fanova doesn't work with multi-index columns
judgment_df = df[dimension_order].droplevel(level=1, axis=1)
y = np.asarray(df["test_acc"]["performance_metric"])

In [ ]:
m_fanova = MultiverseFanova(features=judgment_df, outcome=y)
m_fanova.quantify_individual_importance()

In [ ]:
m_fanova.fanova.get_most_important_pairwise_marginals(n=5)

In [ ]:
vis = visualizer.Visualizer(m_fanova.fanova, m_fanova.configuration_space, directory = "fanova_output")

In [ ]:
vis.plot_marginal(0)

In [ ]:
vis.plot_pairwise_marginal(['sett_exclude_features', 'sett_exclude_subgroups'])